# Étude B3 — Stress testing & traffic light bâlois

**Question de l'étude** : que perdrait le portefeuille de référence dans des scénarios
définis (historiques et hypothétiques), et où en serait le multiplicateur de capital
réglementaire si l'on appliquait le traffic light bâlois aux modèles de VaR des briques 0–2 ?

- Données : snapshot figé `data/cache/` (2026-07-06, incl. benchmark `^STOXX50E`) — offline, reproductible.
- Conventions : chocs en rendements arithmétiques, portefeuille courant (1 M€), perte positive (SPEC.md B3.1).
- Verdicts verrouillés par `tests/test_etude_stress.py` (règle du projet : pas de résultat sans test).
- Sources : Bâle (1996) *Supervisory framework for backtesting* ; BCBS (2018) *Stress testing principles* ; Jorion chap. 14.


In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from riskplatform.backtest import count_exceptions, rolling_traffic_light, traffic_light
from riskplatform.config import load_config
from riskplatform.data import load_returns
from riskplatform.portfolio import portfolio_returns
from riskplatform.reporting import plot_stress_pnl
from riskplatform.stress import run_stress_suite
from riskplatform.var import rolling_var, rolling_var_conditional

REPO = Path.cwd().parent
CACHE = REPO / "data" / "cache"
ASSETS = REPO / "assets"
ALPHA = 0.99
NOTIONAL = 1_000_000.0

config = load_config(REPO / "config" / "portfolio.yaml")
tickers = list(config.portfolio.weights.index)
_, returns = load_returns(tickers, config.portfolio.currencies, config.start, config.end, cache_dir=CACHE)
_, bench_frame = load_returns(
    [config.benchmark_ticker], {config.benchmark_ticker: config.benchmark_currency},
    config.start, config.end, cache_dir=CACHE,
)
benchmark = bench_frame[config.benchmark_ticker]
pnl = portfolio_returns(returns, config.portfolio.weights)
print(f"{len(pnl)} rendements de portefeuille, {pnl.index.min().date()} -> {pnl.index.max().date()}")

3112 rendements de portefeuille, 2014-01-03 -> 2026-07-02


## 1. Suite de stress

Le catalogue par défaut (SPEC.md B3.5) : deux fenêtres historiques datées (COVID, taux 2022),
la **pire fenêtre 20 j extraite des données** (l'extraction prouve où est le pire épisode au
lieu de le supposer), deux chocs de prix (uniforme −20 %, Tech US −30 %), un choc d'indice
−15 % propagé par bêtas OLS vs `^STOXX50E`, et trois chocs de paramètres (σ×2, ρ→1, combiné).

Chaque perte est rapportée à la VaR 99 % 1 j historique plein échantillon (« combien de jours
de VaR ») et au **proxy de capital IMA `3·√10·VaR`** (multiplicateur plancher, horizon 10 j en √t).

In [2]:
suite = run_stress_suite(returns, config.portfolio.weights, notional=NOTIONAL,
                         benchmark_returns=benchmark, alpha=ALPHA)
print(f"VaR 99 % 1 j (réf.)      : {suite.var_ref:>10,.0f} EUR")
print(f"Proxy capital 3·√10·VaR : {suite.capital_ref:>10,.0f} EUR")
print(f"Pire scénario            : {suite.worst}")
suite.pnl_table.round(3)

VaR 99 % 1 j (réf.)      :     32,832 EUR
Proxy capital 3·√10·VaR :    311,469 EUR
Pire scénario            : Pire fenetre 20 j (2020-02-20 -> 2020-03-18)


,kind,loss_eur,pct_notional,ratio_var,ratio_capital
scenario,,,,,
COVID-19 (19/02-18/03/2020),historical,369140.766,0.369,11.243,1.185
Hausse des taux 2022 (03/01-12/10),historical,156266.814,0.156,4.760,0.502
Actions uniformes -20 %,price,200000.000,0.200,6.092,0.642
Tech US -30 %,price,112500.000,0.112,3.427,0.361
Euro Stoxx 50 -15 % (betas),index,127814.615,0.128,3.893,0.410
Pire fenetre 20 j (2020-02-20 -> 2020-03-18),historical,380296.607,0.380,11.583,1.221


In [3]:
suite.risk_table.round(1)

,var_base,var_stressed,es_stressed,ratio
scenario,,,,
Volatilites x2,28907.5,57815.0,66236.6,2.0
Correlations -> 1,28907.5,45449.8,52070.3,1.6
"Crise systemique (sigma x2, rho -> 1)",28907.5,90899.7,104140.5,3.1


**Lecture.**

- **Le replay COVID perd 369 k€ (36,9 % du notional), soit 11,2 jours de VaR — et 1,19× le
  proxy de capital** : un scénario historique réellement advenu dépasse ce que le capital
  calibré sur la VaR couvrirait. C'est l'argument d'existence du stress testing (la VaR dit
  le quantile à 1 %, pas la taille de ce qui advient au-delà, et √10 suppose la diffusion i.i.d.).
- La **pire fenêtre 20 j extraite** (20/02 → 18/03/2020, 380 k€) coïncide à un jour près avec
  la fenêtre COVID datée dans la spec : l'extraction valide le choix des dates.
- **Chocs de paramètres** : ρ→1 multiplie la VaR par **1,57** — la diversification vaut ~36 %
  de la VaR de ce portefeuille, c'est elle que le choc tue. σ×2 double exactement la VaR
  (homogénéité, sanity check) ; le combiné « crise systémique » est le produit des deux (×3,14).
- **Choc d'indice −15 %** : perte 12,8 % du notional, soit un bêta moyen pondéré ≈ 0,85 (le
  bloc USD est peu corrélé au Stoxx). *Limite (amendement #5)* : bêtas OLS pleine période —
  en crise les bêtas montent avec les corrélations, la propagation est donc sous-estimée ;
  le choc ρ→1 capture cet effet par ailleurs.

In [4]:
fig = plot_stress_pnl(suite.pnl_table, suite.var_ref, suite.capital_ref,
                      out_path=str(ASSETS / "etude_stress_pnl.png"))
suite.pnl_by_position.round(0)

,TTE.PA,MC.PA,SAN.PA,BNP.PA,AIR.PA,AAPL,MSFT,NVDA
scenario,,,,,,,,
COVID-19 (19/02-18/03/2020),-64045.0,-36963.0,-23490.0,-63513.0,-77916.0,-29882.0,-32506.0,-40825.0
Hausse des taux 2022 (03/01-12/10),22508.0,-16841.0,-5519.0,-31511.0,-19772.0,-10833.0,-26401.0,-67898.0
Actions uniformes -20 %,-25000.0,-25000.0,-25000.0,-25000.0,-25000.0,-25000.0,-25000.0,-25000.0
Tech US -30 %,0.0,0.0,0.0,0.0,0.0,-37500.0,-37500.0,-37500.0
Euro Stoxx 50 -15 % (betas),-16888.0,-19942.0,-11096.0,-23581.0,-23045.0,-9358.0,-9442.0,-14461.0
Pire fenetre 20 j (2020-02-20 -> 2020-03-18),-64526.0,-39106.0,-23803.0,-63605.0,-78466.0,-31626.0,-33163.0,-46002.0


**Table par position** : le scénario taux 2022 raconte la rotation sectorielle — TotalEnergies
finit **positif** (+22,5 k€, année de choc énergétique) pendant que NVDA perd 68 k€ ; en mars
2020 en revanche, tout perd en même temps (Airbus −78 k€ en tête) : c'est la différence entre
un choc sectoriel et un choc systémique, visible position par position.

## 2. Traffic light bâlois (99 %, fenêtre 250 j)

Zones dérivées de la CDF binomiale `B(250, 1 %)` : verte 0–4 exceptions (`P(X≤k) < 95 %`),
jaune 5–9, rouge ≥ 10 (`P(X≤k) ≥ 99,99 %`) — bornes retrouvées par le code, pas codées en dur.
Multiplicateur de capital = 3 + plus-factor (0 en vert, 0,40→0,85 en jaune, 1,00 en rouge).

On applique le compte glissant à trois modèles : **paramétrique 250 j** (inconditionnelle, B0),
**EWMA-t** et **GARCH-t** (conditionnelles Student-t, B2), sur toutes les dates prévues du snapshot.

In [5]:
models = {
    "Paramétrique 250 j": rolling_var(pnl, "parametric", alpha=ALPHA, window=250),
    "EWMA-t": rolling_var_conditional(pnl, "ewma", alpha=ALPHA, window=1000,
                                      refit_every=20, dist="student"),
    "GARCH-t": rolling_var_conditional(pnl, "garch", alpha=ALPHA, window=1000,
                                       refit_every=20, dist="student"),
}
zones = {}
rows = []
for name, var_series in models.items():
    exceptions = count_exceptions(pnl.loc[var_series.index], var_series)
    rolling = rolling_traffic_light(exceptions, alpha=ALPHA, window=250)
    last = traffic_light(exceptions, alpha=ALPHA, window=250)
    zones[name] = rolling
    shares = rolling["zone"].value_counts(normalize=True)
    red_dates = rolling.index[rolling["zone"] == "red"]
    rows.append({
        "modèle": name,
        "% vert": round(100 * shares.get("green", 0.0), 1),
        "% jaune": round(100 * shares.get("yellow", 0.0), 1),
        "% rouge": round(100 * shares.get("red", 0.0), 1),
        "max exc/250j": int(rolling["n_exceptions"].max()),
        "période rouge": (f"{red_dates.min():%m/%Y} -> {red_dates.max():%m/%Y}"
                          if len(red_dates) else "jamais"),
        "zone actuelle": last["zone"],
        "multiplicateur": last["multiplier"],
    })
pd.DataFrame(rows).set_index("modèle")

,% vert,% jaune,% rouge,max exc/250j,période rouge,zone actuelle,multiplicateur
modèle,,,,,,,
Paramétrique 250 j,45.3,36.2,18.5,15,11/2018 -> 02/2021,green,3.0
EWMA-t,52.6,43.0,4.4,11,03/2020 -> 08/2020,green,3.0
GARCH-t,62.3,33.4,4.3,11,03/2020 -> 08/2020,green,3.0


In [6]:
fig, axes = plt.subplots(len(zones), 1, figsize=(11, 9), sharex=True, sharey=True)
for ax, (name, rolling) in zip(axes, zones.items()):
    top = 16.5
    ax.axhspan(-0.5, 4.5, color="tab:green", alpha=0.12)
    ax.axhspan(4.5, 9.5, color="gold", alpha=0.15)
    ax.axhspan(9.5, top, color="tab:red", alpha=0.12)
    ax.plot(rolling.index, rolling["n_exceptions"], color="black", linewidth=1.2)
    ax.set_ylim(-0.5, top)
    ax.set_ylabel("exceptions / 250 j")
    ax.set_title(name, loc="left", fontsize=10)
    ax.grid(True, alpha=0.2)
axes[0].figure.suptitle("Traffic light bâlois — VaR 99 %, fenêtre glissante 250 j", y=0.995)
fig.tight_layout()
fig.savefig(ASSETS / "etude_traffic_light.png", dpi=110)
plt.show()

**Verdicts (verrouillés par `tests/test_etude_stress.py`).**

- **La paramétrique 250 j passe au rouge dès novembre 2018** (la vol du T4 2018, avant même le
  COVID — max **15 exceptions/250 j** en janvier 2019) et n'en sort qu'en **février 2021** :
  ~483 jours ouvrés en zone rouge (18,5 % de l'historique), multiplicateur plein 4,0 — soit
  **+33 % de capital pendant plus de deux ans**, et un modèle qu'un superviseur aurait
  probablement fait retirer.
- **EWMA-t et GARCH-t touchent aussi le rouge** — nuance honnête vs l'attendu de la spec
  (« les conditionnels-t restent verts/jaunes ») : 81 jours, de mars à août 2020, max 11
  exceptions. Ce sont les **sauts jour-1 depuis un régime calme** diagnostiqués en B2 :
  aucun filtre à retard 1 ne les évite. La différence est la **vitesse de sortie** : la VaR
  conditionnelle remonte avec σ_t et les exceptions s'arrêtent (sortie du rouge en 5 mois,
  vs 27 mois pour la paramétrique).
- Classement : GARCH-t passe 62 % des dates en vert, EWMA-t 53 %, la paramétrique 45 %.
  **Aujourd'hui (fin de snapshot) les trois modèles sont verts, multiplicateur plancher 3,0.**

**Conclusion de la brique.** Le stress testing et le traffic light sont les deux réponses
réglementaires aux limites mesurées dans les briques 0–2 : le premier chiffre ce que la VaR
ne voit pas (COVID = 1,19× le proxy de capital), le second transforme les échecs de backtest
en coût en capital — et il départage les modèles exactement comme nos tests de Kupiec/
Christoffersen : la vol conditionnelle ne supprime pas les exceptions de crise, elle évite
qu'elles s'accumulent.